# Preprocessing Validation

* We want to test two different things here:
    * applying a configuration to multiple scans of the same volume
    * applying a configuration to different center scans of different volumes

We want to see if within a given volume, if these measurements stay similar.
We also want to see if the fast progressors differ significantly from the slow progressors.

The schema to test will be:
* Load E2E volume and select central B-scan
* Flatten to Bruch's membrane (BM)
* Crop 150-pixel sub-BM ROI
* Whole-ROI percentile normalization (1–99%)
* Extract profile at 50 pixels below BM
* Average intensities across ±4 neighboring rows (9-row mean profile)

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# Reusable project functions
from src.barcode.data import (
    load_e2e_volume,
)

# Locate project root
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the project root containing the src folder."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

# Input and output directories
E2E_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "heyex"
    / "meta"
)

VALIDATION_OUTPUT_DIRECTORY = (
    PROJECT_ROOT
    / "results"
    / "preprocessing_validation"
)

VALIDATION_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

# Progression groups
FAST_PROGRESSOR_IDS = [
    8,
    9,
    12,
    41,
    49,
]

SLOW_PROGRESSOR_IDS = [
    17,
    23,
    35,
    36,
    47,
]

PROGRESSION_GROUPS = {
    "fast": FAST_PROGRESSOR_IDS,
    "slow": SLOW_PROGRESSOR_IDS,
}

# Fixed preprocessing configuration
PREPROCESSING_CONFIG = {
    # B-scan selection
    "scan_selection": "center",

    # Flattening
    "bm_layer_name": "BM",
    "reference_row": None,

    # Sub-BM crop
    "depth_below_bm": 150,

    # Intensity normalization
    "normalize": True,
    "normalization_scope": "whole_roi",
    "normalization_center_row": None,
    "normalization_margin": 0,
    "lower_percentile": 1.0,
    "upper_percentile": 99.0,

    # Profile extraction
    "profile_depth": 50,
    "profile_margin": 4,
    "aggregation": "mean",
    "stepsize": 1.0,

    # Output behavior
    "plot_profiles": False,
    "return_profile_data": True,
}


# Construct E2E file registry
volume_registry = []

for progression_group, subject_ids in PROGRESSION_GROUPS.items():
    for subject_id in subject_ids:
        e2e_path = (
            E2E_DIRECTORY
            / f"ea{subject_id}.E2E"
        )

        volume_registry.append(
            {
                "subject_id": subject_id,
                "progression_group": progression_group,
                "e2e_path": e2e_path,
                "file_exists": e2e_path.exists(),
            }
        )

volume_registry = pd.DataFrame(
    volume_registry
)


# Initial validation
print("Project root:")
print(PROJECT_ROOT)

print("\nE2E directory:")
print(E2E_DIRECTORY)

print("\nValidation output directory:")
print(VALIDATION_OUTPUT_DIRECTORY)

print("\nPreprocessing configuration:")
for parameter, value in PREPROCESSING_CONFIG.items():
    print(f"  {parameter}: {value}")

print("\nVolume registry:")
display(volume_registry)

missing_files = volume_registry.loc[
    ~volume_registry["file_exists"],
    "e2e_path",
].tolist()

if missing_files:
    print("\nWarning: the following E2E files were not found:")

    for missing_path in missing_files:
        print(f"  {missing_path}")

else:
    print(
        "\nAll 10 E2E volumes were located successfully."
    )

## Within-Volume Validation

A single E2E volume is selected and the fixed preprocessing configuration is
applied across its B-scans. The processed sub-BM region and extracted intensity
profile are displayed on aligned horizontal axes.

The left and right arrow keys move between adjacent B-scans, allowing visual
assessment of whether the profile and derived measurements remain consistent
across neighboring scans from the same volume.

In [ ]:
# Select and load one E2E volume
VOLUME_ID = 8

selected_volume_path = (
    E2E_DIRECTORY
    / f"ea{VOLUME_ID}.E2E"
)

if not selected_volume_path.exists():
    raise FileNotFoundError(
        f"E2E volume not found: {selected_volume_path}"
    )

selected_volume = load_e2e_volume(
    selected_volume_path
)

print("Volume ID:", VOLUME_ID)
print("Path:", selected_volume_path)
print("Volume shape:", selected_volume.shape)
print("Number of B-scans:", len(selected_volume))
print(
    "Central B-scan:",
    len(selected_volume) // 2,
)

In [ ]:
# Install once if needed:
# %pip install ipympl

%matplotlib widget

from src.barcode.volume_browser import (
    create_volume_profile_browser,
)

volume_browser = create_volume_profile_browser(
    volume=selected_volume,
    initial_index=len(selected_volume) // 2,
    bm_layer_name=PREPROCESSING_CONFIG[
        "bm_layer_name"
    ],
    depth_below_bm=PREPROCESSING_CONFIG[
        "depth_below_bm"
    ],
    reference_row=PREPROCESSING_CONFIG[
        "reference_row"
    ],
    normalize=PREPROCESSING_CONFIG[
        "normalize"
    ],
    normalization_scope=PREPROCESSING_CONFIG[
        "normalization_scope"
    ],
    normalization_center_row=PREPROCESSING_CONFIG[
        "normalization_center_row"
    ],
    normalization_margin=PREPROCESSING_CONFIG[
        "normalization_margin"
    ],
    lower_percentile=PREPROCESSING_CONFIG[
        "lower_percentile"
    ],
    upper_percentile=PREPROCESSING_CONFIG[
        "upper_percentile"
    ],
    profile_depth=PREPROCESSING_CONFIG[
        "profile_depth"
    ],
    profile_margin=PREPROCESSING_CONFIG[
        "profile_margin"
    ],
    aggregation=PREPROCESSING_CONFIG[
        "aggregation"
    ],
    stepsize=PREPROCESSING_CONFIG[
        "stepsize"
    ],
    gray_value_limits=(0, 300),
)

volume_browser.show()

## Between-Volume Validation

The fixed preprocessing configuration is next applied to the center B-scan
from each volume in a selected progression group.

The center scan is calculated separately for every E2E volume using `n // 2`.
This allows the comparison to include volumes with different numbers of
B-scans rather than assuming that every volume contains 97 scans.

The left and right arrow keys move between subjects while preserving the same
preprocessing, normalization, and profile-extraction configuration.

In [ ]:
# ============================================================
# Select progression group
# ============================================================

SELECTED_PROGRESSION_GROUP = "fast"

valid_progression_groups = {
    "fast",
    "slow",
}

if (
    SELECTED_PROGRESSION_GROUP
    not in valid_progression_groups
):
    raise ValueError(
        "SELECTED_PROGRESSION_GROUP must be "
        "'fast' or 'slow'."
    )

selected_registry = (
    volume_registry.loc[
        (
            volume_registry[
                "progression_group"
            ]
            == SELECTED_PROGRESSION_GROUP
        )
        & volume_registry[
            "file_exists"
        ]
    ]
    .sort_values(
        "subject_id"
    )
    .reset_index(
        drop=True
    )
)

if selected_registry.empty:
    raise RuntimeError(
        "No available E2E files were found for "
        f"the '{SELECTED_PROGRESSION_GROUP}' group."
    )

print(
    "Selected progression group:",
    SELECTED_PROGRESSION_GROUP,
)

print(
    "Number of available volumes:",
    len(selected_registry),
)

display(
    selected_registry
)

In [ ]:
# ============================================================
# Construct cohort browser records
# ============================================================

selected_volume_records = [
    {
        "subject_id": int(
            row.subject_id
        ),
        "progression_group": (
            row.progression_group
        ),
        "e2e_path": Path(
            row.e2e_path
        ),
    }
    for row in selected_registry.itertuples(
        index=False
    )
]

selected_volume_records

In [ ]:
# Interactive Matplotlib backend required for arrow-key navigation.
# Install once if needed:
# %pip install ipympl

%matplotlib widget

from src.barcode.cohort_browser import (
    create_cohort_center_profile_browser,
)

cohort_browser = (
    create_cohort_center_profile_browser(
        volume_records=selected_volume_records,
        initial_position=0,
        bm_layer_name=PREPROCESSING_CONFIG[
            "bm_layer_name"
        ],
        depth_below_bm=PREPROCESSING_CONFIG[
            "depth_below_bm"
        ],
        reference_row=PREPROCESSING_CONFIG[
            "reference_row"
        ],
        normalize=PREPROCESSING_CONFIG[
            "normalize"
        ],
        normalization_scope=(
            PREPROCESSING_CONFIG[
                "normalization_scope"
            ]
        ),
        normalization_center_row=(
            PREPROCESSING_CONFIG[
                "normalization_center_row"
            ]
        ),
        normalization_margin=(
            PREPROCESSING_CONFIG[
                "normalization_margin"
            ]
        ),
        lower_percentile=(
            PREPROCESSING_CONFIG[
                "lower_percentile"
            ]
        ),
        upper_percentile=(
            PREPROCESSING_CONFIG[
                "upper_percentile"
            ]
        ),
        profile_depth=(
            PREPROCESSING_CONFIG[
                "profile_depth"
            ]
        ),
        profile_margin=(
            PREPROCESSING_CONFIG[
                "profile_margin"
            ]
        ),
        aggregation=(
            PREPROCESSING_CONFIG[
                "aggregation"
            ]
        ),
        stepsize=(
            PREPROCESSING_CONFIG[
                "stepsize"
            ]
        ),
        gray_value_limits=(
            0,
            300,
        ),
    )
)

cohort_browser.show()